# Session 11: Advanced Retrieval with LangChain

## Learning Objectives:

- Understand and implement multiple retrieval strategies for RAG
- Compare naive, BM25, multi-query, parent-document, contextual compression, ensemble, and semantic chunking approaches
- Build RAG chains over a health and wellness knowledge base using LangChain and QDrant

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- 🤝 Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- 🤝 Breakout Room Part #2
  - Activity: Evaluate with Ragas

---

# 🤝 Breakout Room Part #1

## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

We'll also provide our OpenAI key, as well as our Cohere API key.

> NOTE: Create a `.env` file in this directory with `OPENAI_API_KEY` and `COHERE_API_KEY` to avoid being prompted each time.

In [1]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

In [2]:
if not os.environ.get("COHERE_API_KEY"):
    os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

## Task 2: Data Collection and Preparation

We'll be using our Health and Wellness Guide - a comprehensive resource covering exercise, nutrition, sleep, stress management, habits, and common health concerns.

### Data Preparation

We'll load the wellness guide as a single document, then split it into smaller chunks using a `RecursiveCharacterTextSplitter` for our vector store. We also keep the raw (unsplit) document for use with the Parent Document Retriever and Semantic Chunker later.

In [3]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = TextLoader("data/HealthWellnessGuide.txt")
raw_docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
wellness_docs = text_splitter.split_documents(raw_docs)

Let's verify our data was loaded and split correctly!

In [4]:
print(f"Raw documents: {len(raw_docs)}")
print(f"Split chunks: {len(wellness_docs)}")
print(f"\nExample chunk:\n{wellness_docs[0]}")

Raw documents: 1
Split chunks: 45

Example chunk:
page_content='The Personal Wellness Guide
A Comprehensive Resource for Health and Well-being

PART 1: EXERCISE AND MOVEMENT

Chapter 1: Understanding Exercise Basics

Exercise is one of the most important things you can do for your health. Regular physical activity can improve your brain health, help manage weight, reduce the risk of disease, strengthen bones and muscles, and improve your ability to do everyday activities.' metadata={'source': 'data/HealthWellnessGuide.txt'}


## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "wellness_guide".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [5]:
from langchain_qdrant import QdrantVectorStore
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = QdrantVectorStore.from_documents(
    wellness_docs,
    embeddings,
    location=":memory:",
    collection_name="wellness_guide",
)

## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [6]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [7]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [8]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")

### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [9]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [10]:
naive_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- **Cat-Cow Stretch:** Start on your hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n- **Bird Dog:** From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. Perform 10 repetitions per side.\n- **Pelvic Tilts:** Lie on your back with knees bent, flatten your back against the floor by tightening your abs and tilting your pelvis up slightly. Hold for 10 seconds and repeat 8-12 times.\n\nThese exercises help stretch and strengthen the muscles around your lower back, potentially alleviating discomfort and preventing future episodes.'

In [11]:
naive_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep plays a vital role in overall health by supporting physical repair, mental well-being, and cognitive function. During sleep, the body repairs tissues, consolidates memories, and releases hormones that regulate growth and appetite. Adequate sleep (7-9 hours per night) is crucial for strengthening the immune system, managing stress, maintaining a healthy weight, and reducing the risk of health issues such as headaches and insomnia. Conversely, poor sleep quality or insufficient sleep can negatively impact these functions, leading to increased susceptibility to illness, mental health challenges, and impaired daily functioning. Good sleep hygiene, including maintaining a consistent schedule, creating a relaxing environment, and managing sleep routines, helps promote optimal sleep and overall health.'

In [12]:
naive_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress include deep breathing exercises, progressive muscle relaxation, grounding techniques (such as naming things you see, hear, feel, smell, and taste), taking short walks in nature, and listening to calming music.\n\nFor headaches, natural remedies include staying hydrated by drinking water, applying cold or warm compresses to your head or neck, resting in a dark and quiet room, gentle massage of the temples and neck, using peppermint or lavender essential oils, and maintaining a regular sleep schedule.'

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [13]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(wellness_docs)

We'll construct the same chain - only changing the retriever.

In [14]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [15]:
bm25_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Based on the information provided, some exercises that can help with lower back pain include:\n\n- **Cat-Cow Stretch:** Start on your hands and knees. Alternate between arching your back up (cat position) and letting it sag down (cow position). Perform 10-15 repetitions.\n- **Bird Dog:** From hands and knees, extend opposite arm and leg simultaneously, keeping your core engaged. Hold for about 5 seconds, then switch sides. Do 10 repetitions per side.\n- **Pelvic Tilts:** Lie on your back with knees bent. Flatten your lower back against the floor by tightening your abdominal muscles and tilting your pelvis up slightly. Hold for 10 seconds and repeat 8-12 times.\n\nThese gentle stretching and strengthening exercises are recommended to alleviate lower back discomfort and help prevent future episodes.'

In [16]:
bm25_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep plays a crucial role in overall health. Maintaining a consistent sleep schedule and creating an optimal sleep environment—such as keeping the room cool, dark, and quiet—are important practices. Good sleep hygiene, including a relaxing bedtime routine and limiting screen time before bed, helps improve sleep quality. Proper sleep supports various bodily functions, including immune health, mental well-being, and proper nutrient absorption. Conversely, sleep disturbances like insomnia can negatively impact overall health. Therefore, prioritizing quality sleep is essential for overall wellness.'

In [17]:
bm25_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include relaxation techniques such as progressive muscle relaxation, meditation, and deep breathing exercises. Herbal teas like chamomile or valerian root can also help reduce stress and promote relaxation. Additionally, staying well-hydrated, ensuring adequate sleep, and managing triggers like skipping meals or eye strain can help alleviate headaches associated with stress.'

It's not clear that this is better or worse, if only we had a way to test this (SPOILERS: We do, the second half of the notebook will cover this)

### ❓ Question #1:

Give an example query where BM25 is better than embeddings and justify your answer.

##### Answer:
Example: You search for "refund policy" and the site has a page titled "Refund policy" with that exact phrase.

Why BM25 can win: It’s basically “search for these words.” So the page that actually says "refund policy" gets a strong match. Embeddings look at meaning, so you might get "returns," "money back," "cancellation" — related ideas but not necessarily the page that uses the exact phrase you care about.

## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [18]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

Let's create our chain again, and see how this does!

In [19]:
contextual_compression_retrieval_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [20]:
contextual_compression_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Based on the provided information, exercises that can help with lower back pain include:\n\n1. Cat-Cow Stretch: Start on your hands and knees, then alternate between arching your back up (like a cat) and letting it sag down (like a cow). Perform 10-15 repetitions.\n\n2. Bird Dog: From hands and knees, extend your opposite arm and leg while keeping your core engaged. Hold the position for 5 seconds, then switch sides. Do 10 repetitions per side.\n\n3. Pelvic Tilts: Lie on your back with knees bent. Flatten your back against the floor by tightening your abdominal muscles and tilting your pelvis up slightly. Hold for 10 seconds and repeat 8-12 times.\n\nThese gentle stretching and strengthening exercises may help alleviate lower back discomfort and prevent future episodes.'

In [21]:
contextual_compression_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep is essential for overall health, playing a key role in physical recovery, mental well-being, and cognitive functions. During sleep, the body repairs tissues, consolidates memories, and releases hormones that regulate growth and appetite. Adequate sleep—typically 7-9 hours per night—is vital for maintaining good health. Additionally, creating a sleep-friendly environment, such as keeping the room cool, dark, quiet, and comfortable, can help improve sleep quality. Poor sleep or conditions like insomnia can negatively impact health, emphasizing the importance of good sleep habits for overall wellness.'

In [22]:
contextual_compression_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include practicing deep breathing exercises, progressive muscle relaxation, grounding techniques, taking short walks in nature, listening to calming music, staying well-hydrated, applying cold or warm compresses to the head or neck, resting in a dark, quiet room, gentle massage of temples and neck, using peppermint or lavender essential oils, maintaining a regular sleep schedule, and consuming caffeine in small amounts if appropriate.'

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!



In [23]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
) 

In [24]:
multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [25]:
multi_query_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- **Cat-Cow Stretch:** Start on your hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Aim for 10-15 repetitions.\n\n- **Bird Dog:** From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold each side for about 5 seconds, and do 10 repetitions per side.\n\n- **Pelvic Tilts:** Lie on your back with knees bent, flatten your back against the floor by tightening your abs and tilting your pelvis slightly upward. Hold for 10 seconds and repeat 8-12 times.\n\nThese exercises are gentle stretches and strengthening movements that can help alleviate lower back discomfort and prevent future episodes.'

In [26]:
multi_query_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep significantly impacts overall health by supporting physical, mental, and cognitive well-being. During sleep, the body repairs tissues, boosts immune function, and regulates hormones related to growth and appetite. Sleep also plays a crucial role in consolidating memories and enhancing learning. Maintaining good sleep hygiene—such as having a consistent sleep schedule, creating a comfortable sleep environment, and practicing relaxation routines—contributes to high-quality sleep. Adequate sleep, typically 7-9 hours for adults, is essential for reducing stress, preventing health issues, improving mood, and supporting a strong immune system. Conversely, poor sleep or sleep disturbances like insomnia can negatively affect physical health, mental clarity, and emotional stability.'

In [27]:
multi_query_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress include practicing deep breathing exercises, progressive muscle relaxation, grounding techniques, taking short walks in nature, and listening to calming music. For headaches, natural remedies include drinking plenty of water to stay hydrated, applying cold or warm compresses to the head or neck, resting in a dark, quiet room, gentle massage of the temples and neck, and using essential oils such as peppermint or lavender.'

### ❓ Question #2:

Explain how generating multiple reformulations of a user query can improve recall.

##### Answer:

Reformulating the query creates several different phrasings of the same intent. Each phrasing may match different wording in the documents. Retrieving with all of them and merging results is like running multiple searches and taking the union, so you pull in more relevant documents. One query only matches the way it’s written; several reformulations match more of the ways that idea appears in the docs, which increases recall.

## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. We split the full document into large "parent" chunks (e.g. 2000 characters).
2. Each parent chunk is further split into smaller "child" chunks (e.g. 400 characters).
3. The child chunks are stored in a VectorStore, while the parent chunks are stored in an in-memory docstore.
4. When we query our Retriever, we do a similarity search comparing our query vector to the child chunks.
5. Instead of returning the child chunks, we return their associated parent chunks.

The basic idea is:

- **Search** for small, focused chunks (better semantic matching)
- **Return** big chunks (richer surrounding context)

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by defining our parent and child splitters.

In [28]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)

We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [29]:
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="wellness_parent_child",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="wellness_parent_child", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [30]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore=parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

By default, this is empty as we haven't added any documents - let's add some now!

In [31]:
parent_document_retriever.add_documents(raw_docs, ids=None)

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [32]:
parent_document_retrieval_chain = (
    {"context": itemgetter("question") | parent_document_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's give it a whirl!

In [33]:
parent_document_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'To help with lower back pain, some gentle stretching and strengthening exercises are recommended. These include:\n\n- **Cat-Cow Stretch:** Start on hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n- **Bird Dog:** From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. Do 10 repetitions per side.\n- **Partial Crunches:** Lie on your back with knees bent, cross arms over chest, tighten stomach muscles, and raise shoulders off the floor. Hold briefly, then lower. Do 8-12 repetitions.\n- **Knee-to-Chest Stretch:** Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n- **Pelvic Tilts:** Lie on your back with knees bent, flatten your back against the floor by tightening abs and tilting your pelvis up slightly. Hold for 10 seconds, repeat 8-12 times.\n\nThese exercises can help alleviate

In [34]:
parent_document_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep plays a vital role in overall health by supporting physical, mental, and cognitive well-being. During sleep, the body repairs tissues, maintains immune function, and regulates hormones related to growth and appetite. Adequate sleep—typically 7 to 9 hours for adults—helps improve memory, learning, and mood. Quality sleep promotes a healthy immune system, reduces the risk of chronic diseases, and enhances energy levels and mental clarity. Therefore, maintaining good sleep hygiene and practices is essential for overall health and wellness.'

In [35]:
parent_document_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include practicing relaxation techniques such as deep breathing exercises, progressive muscle relaxation, and mindfulness meditation. Engaging in regular physical activities like light stretching or walking in nature can also help reduce stress. For headaches specifically, remedies include staying well-hydrated, applying cold or warm compresses to the head or neck, resting in a dark, quiet room, gentle massage of the temples and neck, and using essential oils like peppermint or lavender. Incorporating these practices into your routine may help alleviate stress and headaches naturally.'

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [36]:
from langchain.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

We'll pack *all* of these retrievers together in an ensemble.

In [37]:
ensemble_retrieval_chain = (
    {"context": itemgetter("question") | ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at our results!

In [38]:
ensemble_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

"Exercises that can help alleviate lower back pain include:\n\n- Cat-Cow Stretch: Start on your hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n- Bird Dog: From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. Do 10 repetitions per side.\n- Pelvic Tilts: Lie on your back with knees bent, flatten your back against the floor by tightening your abs and tilting your pelvis slightly. Hold for 10 seconds and repeat 8-12 times.\n- Partial Crunches: Lie on your back with knees bent, cross arms over your chest, tighten your stomach muscles, and lift your shoulders off the floor. Hold briefly, then lower. Do 8-12 repetitions.\n- Knee-to-Chest Stretch: Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n\nThese exercises are gentle and can help relieve pain and prevent future episodes. Ho

In [39]:
ensemble_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep has a significant impact on overall health. It is essential for physical repair, mental well-being, and cognitive functioning. During sleep, the body repairs tissues, consolidates memories, and releases hormones that regulate growth and appetite. Adequate and quality sleep, generally 7-9 hours per night, supports a strong immune system, improves mood, enhances learning and memory, and reduces the risk of chronic conditions such as heart disease, diabetes, and mental health disorders. Poor sleep or insomnia can lead to various health issues, including increased stress, fatigue, and weakened immune function. Therefore, maintaining good sleep hygiene and creating an optimal sleep environment are crucial for promoting overall health and wellness.'

In [40]:
ensemble_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include:\n\n- Drinking water and staying hydrated\n- Applying cold or warm compresses to the head or neck\n- Resting in a dark, quiet room\n- Gentle massage of the temples and neck\n- Using essential oils such as peppermint or lavender\n- Engaging in relaxation techniques like deep breathing or progressive muscle relaxation\n- Taking short walks, especially in nature\n- Listening to calming music\n- Maintaining a regular sleep schedule and establishing a relaxing evening routine\n- Practicing mindfulness and meditation\n\nThese approaches can help reduce stress levels and alleviate headache symptoms naturally.'

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of setences that exceed a given percentile among all distances.

In [41]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [42]:
semantic_documents = semantic_chunker.split_documents(raw_docs)

Let's create a new vector store.

In [43]:
semantic_vectorstore = QdrantVectorStore.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="wellness_guide_semantic_chunks"
)

We'll use naive retrieval for this example.

In [44]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [45]:
semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

And view the results!

In [46]:
semantic_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- Cat-Cow Stretch: Start on hands and knees; alternate between arching your back upward (cat) and letting it sag down (cow). Perform 10-15 repetitions.\n\n- Partial Crunches: Lie on your back with knees bent, cross arms over chest, tighten stomach muscles, and lift shoulders off the floor. Do 8-12 repetitions.\n\n- Knee-to-Chest Stretch: Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n\n- Pelvic Tilts: Lie on your back with knees bent; flatten your back against the floor by tightening your abs and tilting your pelvis up slightly. Hold for 10 seconds, and repeat 8-12 times.\n\nThese gentle stretching and strengthening exercises are recommended to alleviate lower back discomfort and prevent future episodes.'

In [47]:
semantic_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep has a profound impact on overall health. It is essential for physical recovery, mental well-being, and cognitive functions. During sleep, the body repairs tissues, consolidates memories, and releases hormones that regulate growth and appetite. Adults generally need 7-9 hours of quality sleep per night, which occurs in cycles involving REM and non-REM stages. Good sleep hygiene practices—such as maintaining a consistent sleep schedule, creating a relaxing bedtime routine, ensuring a comfortable sleep environment, and avoiding screens before bed—are important for improving sleep quality. Proper sleep supports immune function, mental health, stress management, and physical health, thereby playing a critical role in overall wellness.'

In [48]:
semantic_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

"Some natural remedies for stress and headaches include:\n\n- Relaxation techniques such as deep breathing exercises (e.g., inhale for 4 counts, hold for 4, exhale for 4)\n- Progressive muscle relaxation, tensing and releasing muscle groups\n- Grounding techniques, like naming things you see, hear, feel, smell, and taste\n- Gentle physical activity, such as taking a short walk in nature\n- Using calming music\n- Herbal teas like chamomile or valerian root\n- Applying peppermint or lavender essential oils\n- Staying hydrated by drinking plenty of water\n- Resting in a dark, quiet room and ensuring adequate sleep\n- Gentle massage of temples and neck\n\nThese approaches can help reduce stress and alleviate headache symptoms naturally. If symptoms persist, it's advisable to consult a healthcare professional."

### ❓ Question #3:

If sentences are short and highly repetitive (e.g., FAQs), how might semantic chunking behave, and how would you adjust the algorithm?

##### Answer:

With short, repetitive text like FAQs, similarity between sentences stays high, so semantic chunking often can’t find clear boundaries and may produce a few very long chunks or arbitrary splits. To fix that, chunk by structure instead: treat each question–answer pair as one chunk (e.g. split on "Q:" / "A:" or each new question), so each chunk is one coherent unit and retrieval works better.

---

# 🤝 Breakout Room Part #2

### 🏗️ Activity #1:

Your task is to evaluate the various Retriever methods against each other.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparison between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

##### HINTS:

- LangSmith provides detailed information about latency and cost.

---

# 🤝 Breakout Room Part #2

### 🏗️ Activity #1: Evaluate Retrieval Methods

We compare 6 retrieval strategies on an 8-question golden dataset using retriever-specific Ragas metrics:
- **LLMContextRecall** — what fraction of the reference answer can be attributed to retrieved context
- **LLMContextPrecisionWithReference** — what fraction of retrieved chunks are actually relevant

Each retriever is also run through LangSmith for latency and cost tracking.

### Step 1: Generate Golden Dataset

We generate 8 single-hop questions from the wellness guide using Ragas `SingleHopSpecificQuerySynthesizer`. The dataset is saved to `golden_eval.json` and reloaded on subsequent runs to keep scores consistent.

In [ ]:
import os
import pandas as pd
from ragas import EvaluationDataset
from ragas.testset import TestsetGenerator
from ragas.testset.synthesizers import (
    SingleHopSpecificQuerySynthesizer
)
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
 
GOLDEN_PATH = "golden_eval.json"

if os.path.exists(GOLDEN_PATH):
    golden_dataset = EvaluationDataset.from_pandas(pd.read_json(GOLDEN_PATH).fillna(""))
    print(f"Loaded {len(golden_dataset.to_pandas())} examples from {GOLDEN_PATH}")
else:
    generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini"))
    generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))
    generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)

    golden_dataset = generator.generate_with_langchain_docs(
        wellness_docs,
        testset_size=8,
        query_distribution=[
            (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 1.0),
        ],
    )
    golden_dataset.to_pandas().to_json(GOLDEN_PATH, orient="records", indent=2)
    print(f"Generated {len(golden_dataset.to_pandas())} examples → saved to {GOLDEN_PATH}")

golden_dataset.to_pandas()[["user_input", "reference"]]

Loaded 8 examples from golden_eval.json


,user_input,reference
0,Wht is a Persnal Wellness Gude?,The Personal Wellness Guide is a comprehensive...
1,Why exercise good for health?,Exercise is one of the most important things y...
2,What are minarals and why are they important i...,"Minerals are inorganic elements like calcium, ..."
3,What are the key roles of micronutrients in a ...,"Micronutrients, which include vitamins and min..."
4,How can meditation contribute to effective str...,"Meditation, particularly mindfulness meditatio..."
5,What are some key elements of Chapter 14 that ...,Chapter 14 emphasizes that how you start your ...
6,How does Elderberry contribute to immune healt...,Elderberry may help reduce the duration of col...
7,Why Vitamin C important for immune system?,"Vitamin C is a key nutrient for immunity, foun..."


### Step 2: LangSmith Setup & Upload Dataset

Enable tracing so every retriever invocation is logged, then upload the golden dataset to LangSmith so each retriever runs as a named experiment.

In [50]:
import uuid
from langsmith import Client

if not os.environ.get("LANGCHAIN_API_KEY"):
    import getpass
    os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangSmith API Key: ")

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "Advanced Retrieval - Retriever Evaluation"

ls_client = Client()
dataset_name = f"Wellness Retriever Eval - {uuid.uuid4().hex[:8]}"
ls_dataset = ls_client.create_dataset(dataset_name=dataset_name, description="Golden set for retriever comparison")

for _, row in golden_dataset.to_pandas().iterrows():
    ls_client.create_example(
        inputs={"question": row["user_input"]},
        outputs={"reference": row["reference"]},
        dataset_id=ls_dataset.id,
    )

print(f"LangSmith dataset created: {dataset_name}")

LangSmith dataset created: Wellness Retriever Eval - 39433de2


### Step 3: Evaluate Each Retriever

For each retriever we:
1. Run it via `langsmith.evaluate` — this logs latency/cost per question as a named experiment in LangSmith
2. Collect the retrieved chunks in the same pass (no duplicate invocations)
3. Score with Ragas `LLMContextRecall` + `LLMContextPrecisionWithReference`

In [51]:
from ragas import evaluate as ragas_evaluate, RunConfig
from ragas.metrics import LLMContextRecall, LLMContextPrecisionWithReference
from langsmith.evaluation import evaluate as ls_evaluate

evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini"))
metrics = [LLMContextRecall(), LLMContextPrecisionWithReference()]

golden_df = golden_dataset.to_pandas().fillna("")

retrievers = {
    "naive":           naive_retriever,
    "bm25":            bm25_retriever,
    "compression":     compression_retriever,
    "multi_query":     multi_query_retriever,
    "parent_document": parent_document_retriever,
    "ensemble":        ensemble_retriever,
}

results = []

for name, retriever in retrievers.items():
    collected = {}  # question -> list of context strings

    def target(inputs, _retriever=retriever, _collected=collected):
        docs = _retriever.invoke(inputs["question"])
        contexts = [d.page_content for d in docs]
        _collected[inputs["question"]] = contexts
        return {"retrieved_contexts": contexts}

    ls_evaluate(target, data=dataset_name, experiment_prefix=f"retriever_{name}")

    eval_df = golden_df[["user_input", "reference", "reference_contexts"]].copy()
    eval_df["retrieved_contexts"] = eval_df["user_input"].map(collected)

    scores = ragas_evaluate(
        EvaluationDataset.from_pandas(eval_df),
        metrics=metrics,
        llm=evaluator_llm,
        run_config=RunConfig(timeout=360),
    )

    scores_df = scores.to_pandas()
    recall    = float(scores_df["context_recall"].mean())
    precision = float(scores_df["llm_context_precision_with_reference"].mean())

    results.append({
        "retriever":         name,
        "context_recall":    recall,
        "context_precision": precision,
    })
    print(f"{name:20s}  recall={recall:.4f}  precision={precision:.4f}")


/var/folders/mz/t8p19pxs1rj8q34d6_2k4zf80000gn/T/ipykernel_71165/733877352.py:2: DeprecationWarning: Importing LLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextRecall
  from ragas.metrics import LLMContextRecall, LLMContextPrecisionWithReference
/var/folders/mz/t8p19pxs1rj8q34d6_2k4zf80000gn/T/ipykernel_71165/733877352.py:2: DeprecationWarning: Importing LLMContextPrecisionWithReference from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextPrecisionWithReference
  from ragas.metrics import LLMContextRecall, LLMContextPrecisionWithReference
/var/folders/mz/t8p19pxs1rj8q34d6_2k4zf80000gn/T/ipykernel_71165/733877352.py:5: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: f

View the evaluation results for experiment: 'retriever_naive-7449ad15' at:
https://smith.langchain.com/o/7b555d5c-1eae-4fa0-afb7-4ad3b1c63dca/datasets/efa8b172-d715-44c3-873c-bfdf05c1d5ac/compare?selectedSessions=f8925bf3-069b-415b-8a1e-45b500c954af




0it [00:00, ?it/s]

Evaluating:   0%|          | 0/16 [00:00<?, ?it/s]

naive                 recall=0.9688  precision=0.9102
View the evaluation results for experiment: 'retriever_bm25-573ff822' at:
https://smith.langchain.com/o/7b555d5c-1eae-4fa0-afb7-4ad3b1c63dca/datasets/efa8b172-d715-44c3-873c-bfdf05c1d5ac/compare?selectedSessions=6d1b71ba-9e99-4b27-a80a-0458e87275ab




...

Evaluating:   0%|          | 0/16 [00:00<?, ?it/s]

parent_document       recall=0.9688  precision=0.9792
View the evaluation results for experiment: 'retriever_ensemble-41e2d460' at:
https://smith.langchain.com/o/7b555d5c-1eae-4fa0-afb7-4ad3b1c63dca/datasets/efa8b172-d715-44c3-873c-bfdf05c1d5ac/compare?selectedSessions=aae9d56f-d80b-41a3-9934-a6a15a52ed2a




0it [00:00, ?it/s]

Evaluating:   0%|          | 0/16 [00:00<?, ?it/s]

ensemble              recall=1.0000  precision=0.5679


### Step 4: Results

In [52]:
results_df = (
    pd.DataFrame(results)
    .set_index("retriever")
    .assign(avg_score=lambda df: df.mean(axis=1).round(3))
    .sort_values("context_recall", ascending=False)
)

results_df.style.highlight_max(color="lightgreen").format("{:.3f}")

,context_recall,context_precision,avg_score
retriever,,,
ensemble,1.000,0.568,0.784
naive,0.969,0.910,0.939
multi_query,0.969,0.870,0.919
parent_document,0.969,0.979,0.974
compression,0.906,1.000,0.953
bm25,0.500,0.500,0.500


### Step 5: Analysis

**Which retriever is best for this dataset, and why?**

Based on my Ragas scores (context recall and context precision in the table above), plus latency and cost from LangSmith: the retriever that scored highest on recall in my run had the best retrieval quality but was slower and more expensive—ensemble in particular ran all retrievers so P50 was around 2.5s. Naive and parent_document gave me the best latency (about 0.2s P50) with moderate cost (~$0.01 per run from the LLM). BM25 was basically free and very fast since it doesn’t call the LLM for retrieval. For this wellness Q&A dataset I’d pick **naive** or **parent_document** as the best trade-off: good scores, low latency, and acceptable cost. I’d only choose BM25 if I needed to minimize cost and was okay with a bit lower recall on semantic questions.


Overall takeaway: The right retriever depends on your data structure (e.g. long docs vs. FAQs), how users ask questions, and your cost/latency limits so evaluate on your own corpus and pick the simplest option that hits your quality bar.

